The program can be used to test different embeddings of text, metrics, and LLM responses

## A compact version of RAG test for deployment in HF

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
import os
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
from ordered_set import OrderedSet
#import json
#import requests
#import time
#import textwrap
#import tiktoken
#import bm25s 
#import pandas as pd


import RAG_lib as rlib


In [5]:
if 1:
    # reload rlib if it was changed
    from importlib import reload
    reload(rlib);

## General variables and default options

In [ ]:
# Variables
var_dict = {}
# ~ 1000 words or tokens (~ 2 pages) is a good chunk size for scientific texts
# ~ 500 words (~ 1 page) may be OK too
var_dict['chunk_size'] = 1024 # 512
# overlap between chunks - useful if context is at the boundary of a chunk
var_dict['chunk_overlap'] = 200
var_dict['chunk_overlap_BM25'] = 0 # overlap for BM25 RAG
#chunk_overlap_dict = {'BM25':0} # custom overlap for different RAGs
var_dict['ktop'] =  5 # if 0, then RAG is not used
var_dict['print_costs'] = True # print token usage and costs for LLMs

# BM25, vector
#RAG_representation = 'tokens' # 'tokens', 'vectors'
var_dict['RAG'] = 'words'
#RAG_metric = 'overlap_weighted'
RAG_options = {}
RAG_options['tokens'] = {}
RAG_options['vectors'] = {}

# use a dictionary to query metrics for calculating similarity of embeddings
metric_dict = {'cosine': lambda xs, ys: cosine_similarity(xs, ys)[0]}

# an example of a vector model
RAG_options['vectors']['ollama:nomic-embed-text_cos']  = {}
RAG_options['vectors']['ollama:nomic-embed-text_cos']['embedding']  = \
    OllamaEmbeddings(model="nomic-embed-text").embed_documents
RAG_options['vectors']['ollama:nomic-embed-text_cos']['metric'] = 'cosine'
    

# an example of a token model: words
RAG_options['tokens']['words'] = {}
RAG_options['tokens']['words']['embedding'] = \
    rlib.token_freq_df_embed_documents(tokenizer=rlib.get_words)
RAG_options['tokens']['words']['metric'] = 'token_similarity_weighted'

# an example of a token model: words + BM25
RAG_options['tokens']['BM25'] = {}
RAG_options['tokens']['BM25']['embedding'] = rlib.bm25_embedding
RAG_options['tokens']['BM25']['metric'] = 'BM25_default'

metric_dict['BM25_default'] = rlib.bm25_metric

RAG_dict = {'vectors':['ollama:nomic-embed-text_cos'],
             'tokens':['words', 'BM25']
             }

var_dict['RAG_dict'] = RAG_dict

var_dict['RAGs'] = OrderedSet()
for RAGs in RAG_dict.values():
    var_dict['RAGs'] = var_dict['RAGs'].union(RAGs)

# LLM options
#LLMs = ['gpt-5-mini', 'gpt-5-nano', 'llama3.2', "claude-sonnet-5-5", "gemini-2.5-flash"]
LLM_dict = {'openai':['gpt-5-mini', 'gpt-5-nano'],
             'anthropic':["claude-sonnet-5-5"],
             'google':["gemini-2.5-flash"],
             'hf':[]}

var_dict['LLM_dict'] = LLM_dict

million = 1.e6
var_dict['token_cost_dict'] = \
    {'gpt-5-mini': {'input': 0.25/million, 'output': 2./million},
    'gpt-5-nano': {'input': 0.05/million, 'output': 0.4/million},
    'claude-sonnet-5-5': {'input': 2./million, 'output': 10./million},
    'gemini-2.5-flash': {'input': 0.3/million, 'output': 2.5/million}
    }




var_dict['LLMs'] = OrderedSet()
for LLMs in LLM_dict.values():
    var_dict['LLMs'] = var_dict['LLMs'].union(LLMs)

#hf_models = ["meta-llama/Llama-3.2-3B:auto", "openai/gpt-oss-20b:auto", "Qwen/Qwen3.5-9B:auto"]

var_dict['RAG'] = 'words'
var_dict['LLM'] = 'claude-sonnet-5-5'

/var/folders/mb/yxqk032n4lvgsvnl3lqzbwvr0000gn/T/ipykernel_40681/2018066373.py:27: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  OllamaEmbeddings(model="nomic-embed-text").embed_documents


## Meta data, e.g., cost information

## Environment variables

In [ ]:
# load environment variables from .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
hf_api_key = os.getenv('HF_TOKEN')

if 1:
    # test if api keys exist
    if openai_api_key:
        print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    else:
        print("OpenAI API Key not set")
        
    if anthropic_api_key:
        print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
    else:
        print("Anthropic API Key not set (and this is optional)")
        
    if google_api_key:
        print(f"Google API Key exists and begins {google_api_key[:2]}")
    else:
        print("Google API Key not set (and this is optional)")

    if hf_api_key:
        print(f"HF API Key exists and begins {hf_api_key[:2]}")
    else:
        print("HF API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
HF API Key exists and begins hf


In [ ]:


def openai_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI()
    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response_dict = {'text': response.choices[0].message.content}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        if full_response:
            return response
        else:
            return response_dict
    return chatbot

def google_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI(api_key=google_api_key, 
                 base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response_dict = {'text': response.choices[0].message.content}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot


def anthropic_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = Anthropic()
    def chatbot(messages, full_response=full_response):
        response = bot.messages.create(model=model, messages=messages, max_tokens=1000)
        #print('Anthropic response:', response)
        for r in response.content:
            # return the first text response, skip thinking blocks
            if r.type == 'text':
                response_dict = {'text': r.text}
                break
        usage = response.usage
        # placeholder for usage for anthropic
        response_dict['usage'] = {'prompt_tokens': usage.input_tokens,
                                  'completion_tokens': usage.output_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'output_tokens_details') and usage.output_tokens_details.thinking_tokens:
            reasoning_tokens = usage.output_tokens_details.thinking_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot



def hf_bot(var_dict):
    model = var_dict['LLM']
    full_response = var_dict.get('full_response', 0)
    bot = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=hf_api_key
        )

    def chatbot(messages, full_response=full_response):
        response = bot.chat.completions.create(model=model, messages=messages)
        response = bot.chat.completions.create(
                model=model,  # ":any other providers" is required
                messages=messages)
        response_text = str(response.choices[0].message.content)
        response_dict = {'text': response_text}
        usage = response.usage
        response_dict['usage'] = {'prompt_tokens': usage.prompt_tokens,
                                  'completion_tokens': usage.completion_tokens,
                                  'reasoning_tokens': 0}
        if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
            reasoning_tokens = usage.completion_tokens_details.reasoning_tokens
            response_dict['usage']['reasoning_tokens'] = reasoning_tokens
        return response_dict
    return chatbot


def get_bot(var_dict):
    model = var_dict['LLM']
    LLM_dict = var_dict['LLM_dict']
    if model in LLM_dict.get('openai', []):
        return openai_bot(var_dict)
    elif model in LLM_dict.get('anthropic', []):
        return anthropic_bot(var_dict)
    elif model in LLM_dict.get('google', []):
        return google_bot(var_dict)
    elif model in LLM_dict.get('hf', []):
        return hf_bot(var_dict)
    else:
        return None



## RAG preprocessing

In [64]:
# converte pdf to text and save (if not yet converted)
# generalize later for on-demand additions to RAG
fn0 = 'LAT_FBs-1407.7905v1' # Fermi LAT paper about the Fermi bubbles
pdf_fn = 'RAG_data/%s.pdf' % fn0
txt_fn = 'RAG_data/%s.txt' % fn0
rewrite_txt = False
if not os.path.exists(txt_fn) or rewrite_txt:
    text = rlib.pdf2text(pdf_fn)
    with open(txt_fn, "w", encoding="utf-8") as f:
        f.write(text)
else:
    text = open(txt_fn, "r", encoding="utf-8").read()
var_dict['text'] = text

# calculate the word frequency in the corpus
text_dfs = RAG_options['tokens']['words']['embedding']([text])
word_freq_df = rlib.add_dfs(text_dfs, col_name='frequency')
    

# use the word frequencies as inverse weights
metric_dict['token_similarity_weighted'] = \
    rlib.token_similarity_weighted(word_freq_df)


## Example questions

In [65]:
qfn = 'RAG_data/FB_example_questions.txt'
qs = open(qfn, 'r').readlines()
qs = [q.strip() for q in qs if not q.startswith('#')]

In [66]:

test = 0
if test:
    model_name = 'claude-sonnet-5-5'
    #q = 'How many prime numbers are there between 1 and 100?'
    q = 'size fermi bubbles'
    test_messages = [{"role": "user", "content": q}]
    test_messages.append({"role": "user", "content": RAG_response(q, var_dict)})
    bot = Anthropic()
    response = bot.messages.create(model=model, messages=test_messages, max_tokens=1000)
    
    


In [67]:
if test:
    for i, r in enumerate(response.content):
        print(i, r.type)
        if r.type == 'text':
            response_dict = {'text': r.text}
            break
    print(response_dict)
    response.usage

In [ ]:
if test:
    model = 'claude-sonnet-5-5'
    print(test_messages)
    bot = get_bot(var_dict)
    response = bot(messages=test_messages)
    print(response)

[{'role': 'user', 'content': 'size fermi bubbles'}, {'role': 'user', 'content': '1: flux. We use it to define the direction perpendicular to the edge.\n3. In order to find the position of the edge and its width, we choose a strip along the gradient and\nproject the data onto the gradient direction. This allows us to reduce the fitting to a one-dimensional\nfitting problem. We take the length of the strip to be 40◦(±20◦from the reference point), and\nthe width to be 20◦(±10◦from the reference point). The size of the bins in the projection is 1 .3◦,\nwhich is larger than the pixel size but small enough that resolving the sharp transitions is possible.\n4. Fit the data along the strip with a smoothed step function modeled by the hyperbolic tangent plus\na constant:\nf(φ) = A tanh λ(φ−φ0) + C . (15)\nIn total we have the four parameters A, λ,φ0, and C. Two of them are non-linear ( λandφ0). The\nbest-fit position of the edge is determined by φ0, the width of the edge is defined as ∆ φ= 2/λ,

In [ ]:
#q = 'What is the capital of France?'
#q = 'How many prime numbers are there between 1 and 100?'
q = 'size of fermi bubbles'

test_messages = [{"role": "user", "content": q}]
LLM_dict['hf'] = ["meta-llama/Llama-3.2-1B-Instruct:featherless-ai", # 0
                    "meta-llama/Llama-3.2-3B-Instruct:featherless-ai", # 1
                    "Qwen/Qwen3-0.6B:featherless-ai", # 2
                    "google/gemma-3-4b-it:deepinfra", # 3
                    "Qwen/Qwen3.5-9B:together", # 4
                    "openai/gpt-oss-20b:groq", # 5
                    ]

var_dict['LLMs'] = var_dict['LLMs'].union(LLM_dict['hf'])

model = LLM_dict['hf'][5]
print(q)
print(model)




size of fermi bubbles
openai/gpt-oss-20b:groq


In [72]:
bot = get_bot(var_dict)
response = bot(messages=test_messages)
print('Answer:', response['text'])
print('Usage:', response['usage'])

Answer: **Fermi Bubbles: Size**

The Fermi bubbles are two giant gamma-ray-emitting lobes extending above and below the Milky Way's galactic center.

**Dimensions**
- **Height (extent from the plane):** about 25,000 light-years (~8 kiloparsecs, kpc) each, north and south of the galactic plane
- **Total vertical span:** about 50,000 light-years (~15 kpc) from the tip of one bubble to the tip of the other
- **Width:** each bubble is about 15,000 to 20,000 light-years (~4-6 kpc) across at its widest
- **Angular size:** they extend roughly 50° above and below the galactic plane as seen from Earth

**Context**
- Together they span roughly half the diameter of the Milky Way's stellar disk (~100,000 light-years).
- They are centered on Sagittarius A*, the supermassive black hole at the galactic center.
- They were discovered in 2010 using NASA's Fermi Gamma-ray Space Telescope.

**Possible origin**
The bubbles are thought to come from a past energetic event, such as:
- A period of intense acc

### Test RAGs with different questions

In [73]:
def get_type(model, type_dict):
    for key, models in type_dict.items():
        if model in models:
            return key
    return None


def RAG_response(query, var_dict):
    chunks = var_dict['chunks']
    RAG = var_dict['RAG']
    RAG_dict = var_dict['RAG_dict']
    ktop = var_dict['ktop']
    chunk_overlap = var_dict['chunk_overlap']
    chunk_size = var_dict['chunk_size']
    text = var_dict['text']
    
    if RAG == 'BM25':
        chunk_overlap = var_dict['chunk_overlap_BM25']

    RAG_type = get_type(RAG, RAG_dict)
        
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = splitter.split_text(text)

    embedding = RAG_options[RAG_type][RAG]['embedding']
    metric_name = RAG_options[RAG_type][RAG]['metric']
    metric = metric_dict[metric_name]
    emb_chunks = embedding(chunks)
    emb_qs = embedding([query])

    scores = metric(emb_qs, emb_chunks)
    inds = np.argsort(scores)[::-1][:ktop]

    message = ''
    for i, ind in enumerate(inds):
        message += f'{i + 1}: {chunks[ind]}\n\n'
    return message



    


In [74]:
#RAG = 'ollama:nomic-embed-text_cos'
#RAG_type = rlib.get_type(RAG, RAG_dict)
chunk_overlap = var_dict['chunk_overlap']
if var_dict['RAG'] == 'BM25':
    chunk_overlap = var_dict['chunk_overlap_BM25']


## Split text in chunks

In [75]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=var_dict['chunk_size'],
    chunk_overlap=var_dict['chunk_overlap'],
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(text)
var_dict['chunks'] = chunks
print('The number of chunks is', len(chunks))

The number of chunks is 186


In [76]:
q = qs[1]
print(RAG_response(q, var_dict))

1: the electron spectrum by a power law with an exponential cutoff ∝E−ne−E/E cut. The best fit parameters
aren= 2.17±0.05[stat]+0.33
−0.89[syst] and Ecut= 1.25±0.13[stat]+1.73
−0.68[syst] TeV. The corresponding IC
spectra are shown in Figure 34 on the left. The details of the calculation can be found in Appendix
B.1. The indices and the cutoff values for different foreground models and definitions of the templates
of Loop I and the bubbles are shown in Figure 34 on the right. The bremsstrahlung emission is at least
two orders of magnitude smaller than the IC emission for a characteristic gas density nH≲0.01 cm−3at
a few kpc from the Galactic plane (Snowden et al. 1997), and can be neglected.
We will assume that the center of the bubbles is at b= 25◦, i.e., the distance to the center of
the bubbles is R=R⊙/cosb= 9.4 kpc, where R⊙= 8.5 kpc is the distance to the GC. The total
energy contained in the electron population inside the bubbles above 1 GeV is (1 .0±0.2[stat]+6.0
−1.0[syst]) ×



In [77]:
vars = ['LLM', 'RAG', 'chunk_size', 'chunk_overlap', 'chunk_overlap_BM25', 'ktop', 'print_costs']
special_commands = ['/help', '/LLMs', '/RAGs', '/vars']
var_dict['vars'] = vars
var_dict['special_commands'] = special_commands


In [78]:

def chat(message, history, var_dict=var_dict):
    messages = history + [message]
    if message.startswith('/'):
        response = rlib.execute(message, var_dict=var_dict)
        return response
    if message.find('@') == -1:
        message = '@RAG+LLM: ' + message

    target = message.split(' ')[0]
    query = message.replace(target, '').strip()
    response = ''
    if var_dict['ktop'] > 0 and message.lower().find('rag') > -1:
        # determine RAG response (or context if LLM is used)
        response += f'RAG "{var_dict["RAG"]}" top {var_dict["ktop"]} chunks '
        answer = RAG_response(query, var_dict)
    if message.lower().find('llm') > -1:
        messages = [{"role": "user", "content": query}]
        if len(response) > 0:
            response += '+ '
            messages.append({"role": "user", "content": answer})
        
        
        bot = get_bot(var_dict)
        try:
            bot_response = bot(messages)
            answer = bot_response['text'] # if LLM is used: overwrite answer with LLM response
        except Exception as e:
            bot_response = {}
            answer = f"Error: {e}"
        #print(bot_response)
        
        response += f'LLM "{var_dict["LLM"]}" '
        if var_dict.get('print_costs', False):
            usage_dict = bot_response.get('usage', {})
            costs = rlib.get_token_costs(usage_dict, var_dict)
            #print('Usage dict and costs:',usage_dict, costs)
        else:
            #print('Token costs not printed. Set var_dict["print_costs"] = True to print token costs.')
            costs = None
    response += f'response:\n\n{answer}'
    if costs is not None:
        response += f'\n\n{costs}'

    return response


In [79]:
chat_box = gr.ChatInterface(chat, #type="messages",     
                            textbox=gr.Textbox(placeholder="type /help for available commands", 
                                               lines=1, max_lines=30))
chat_box.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [ ]:
var_dict['print_costs']

True

In [ ]:
get_model_index('nano', var_dict['LLMs'])

Error: NameError: name 'get_model_index' is not defined
[31m---------------------------------------------------------------------------[39m
[31mNameError[39m                                 Traceback (most recent call last)
[36mCell[39m[36m [39m[32mIn[27][39m[32m, line 1[39m
[32m----> [39m[32m1[39m get_model_index([33m'nano'[39m, var_dict[[33m'LLMs'[39m])

[31mNameError[39m: name 'get_model_index' is not defined

Anthropic response: Message(id='msg_011CfrRbcbC9nuinVfWvgZWc', container=None, content=[ThinkingBlock(signature='CAQSlQwKEAgSGAI4AUIIdGhpbmtpbmcSDBEslol5QcLw7NYDSBoMzlK73znO7bIHxkCTIjAglZwQ4sXu7m48OqvT6T5TyNWWlTRodi5VvYKV+wm1bqrXubj1FMlGoJeEPSVqBJAqsgu5dR/AjxN0CLAiQpkLO5h9OnHFHbe9QBuNdLedASiGu9bLTjfRT35nnTLwwpTp3qL9olQfi4n54po73P+3Rat0ReCO74HvPFAHRKpV32vAzHozwUL1OsuNm9GGGksoZvzt1pOqZYuoopIBpirwSDf0XK4tvSrX1PiCg02/tBX4pe/QxvJkQv+0jMJxVzRK7fSqGLof1xaJnQZOsxI7QMnG4NGy5ehAg4bDcYAM7vCgJp8kEiKFHtHy3NUqnPjTI5ItM282pAhqTe9CQfssuGNNBej3GJ9KBVgMI4JqFBItHkoGYJoN4fQmqGnU7JKSyKh+UbLfGHReJ7mwnIw8bqJOOZpUeMdIWREnoAU75UyisPmCiksW3Gx454hP9Bj87mYDayQQ2dXhFIA9tjRLW2JbeNPhUaUehx2fM6saKRbWf2YWCKlX6AWIYPvJEUz05xZXYy/doz+JU8hZdv1WLGQDeyEagH8DvCMvKVgbt3lfQBkt5LQsus2b+rSZ93+8JvX+DPiZ23XGg1DPazHQFCHPIVUoZ+QLWzdWMS2YFvTCY82n7LmE/Ug8+XmdYE1djeTSYjpG61iM9aPpMswHR/WeC4Bn8qoOKJspjiBFQD31z0ZRX8VeNbGoY9xfpVHv3mfueQKz85mVf+KxtRZwDKC1/DR4vVYItGoF5XEt5lQy2NoHsYrC0Urja3LigFarVaIbjbknXBRefs3hJEQcnVDhsOcyL9O4FrKytqc6VJff2r+

In [ ]:
[mdl.find('nano') for mdl in var_dict['LLMs']]